# DLPFC Spatial Domain Clustering

Compare the PCA baseline, PCA + image embeddings (GPFM / UNI / UNI2-h / GigaPath), and precomputed SpaGCN results.


In [ ]:
import os, pickle, warnings, sys
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse
from sklearn.metrics import (
    adjusted_rand_score, normalized_mutual_info_score,
    adjusted_mutual_info_score, homogeneity_score,
    completeness_score, v_measure_score, fowlkes_mallows_score,
)
from sklearn.cluster import KMeans
import scanpy as sc
import torch
warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════
# Import core functions from the synthetic source files instead of redefining them in the notebook
# ══════════════════════════════════════════════════════════
SPG_CLUSTER_DIR = Path("../../synthetic/src/05_downstream/spatial_cluster/DLPFC")
sys.path.insert(0, str(SPG_CLUSTER_DIR.resolve()))
import spagcn_cluster as spg  # build_features / knn_connectivities_sklearn / cluster_with_graph_leiden

print("All dependencies imported successfully")
print(f"spagcn_cluster.py: {SPG_CLUSTER_DIR / 'spagcn_cluster.py'}")


## Configuration

Parameters are read from `configs/spatial_cluster/DLPFC/downstream.yaml` and `configs/common/DLPFC/normalize.yaml`.


In [ ]:
# ══════════════════════════════════════════════════════════
# Load parameters from the synthetic config files
# ══════════════════════════════════════════════════════════
cfg = spg.load_config("configs/spatial_cluster/DLPFC/downstream.yaml")
cluster_cfg = cfg["task3_spatial_cluster"]["clustering"]

# Slice IDs and true number of clusters per slice (from downstream.yaml n_clusters_by_slice)
SLICE_IDS = sorted(cluster_cfg["n_clusters_by_slice"].keys())
N_CLUSTERS_BY_SLICE = {k: int(v) for k, v in cluster_cfg["n_clusters_by_slice"].items()}

# Clustering hyperparameters (from downstream.yaml clustering.*)
SEEDS = cluster_cfg.get("seeds", [42])       # Main-line seeds: [42]
PCA_N = cluster_cfg.get("pca_n_components", 50)
SPATIAL_WEIGHT = cluster_cfg.get("spatial_weight", 1.0)
KNN_K = cluster_cfg.get("adj_k", 6)           # Code default
LEIDEN_RES = cluster_cfg.get("resolution", 1.0)
EMBED_RESOLUTION = LEIDEN_RES  # PCA+embedding methods use a low resolution to avoid over-clustering; can be set to 0.2
KNN_BACKEND = cluster_cfg.get("knn_backend", "sklearn")

# HVG parameters (from configs/common/DLPFC/normalize.yaml)
hvg_cfg = spg.load_config("configs/common/DLPFC/normalize.yaml")
HVG_TOP_GENES = hvg_cfg.get("preprocessing", {}).get("DLPFC", {}).get("n_top_genes", 3000)

# Evaluation metrics (from the downstream.yaml metrics list)
# Note: sklearn functions cannot be serialized from YAML, so map by name here
METRIC_NAME_TO_FUNC = {
    "ari": adjusted_rand_score,
    "nmi": lambda t,p: normalized_mutual_info_score(t,p,average_method="arithmetic"),
    "ami": lambda t,p: adjusted_mutual_info_score(t,p,average_method="arithmetic"),
    "homogeneity": homogeneity_score,
    "completeness": completeness_score,
    "v_measure": v_measure_score,
    "fowlkes_mallows": fowlkes_mallows_score,
}
METRIC_RENAME = {
    "ari": "ARI", "nmi": "NMI", "ami": "AMI",
    "homogeneity": "Homogeneity", "completeness": "Completeness",
    "v_measure": "V-Measure", "fowlkes_mallows": "FM",
}
metric_names_from_cfg = [m.lower() for m in cfg.get("task3_spatial_cluster", {}).get("metrics", ["ari", "nmi", "ami", "homogeneity", "completeness", "v_measure", "fowlkes_mallows"])]
METRIC_FUNCS = {METRIC_RENAME[m]: METRIC_NAME_TO_FUNC[m] for m in metric_names_from_cfg if m in METRIC_NAME_TO_FUNC}

# Image embedding filename suffix (PFM embedding cache format)
PKL_SUFFIX = "112"
SPAGCN_SUFFIX = f"r{PKL_SUFFIX}"

# Data and output paths
DATA_DIR = Path("../../data/DLPFC")
OUTPUT_DIR = Path("../outputs/spatial_cluster_comparison").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Config: configs/spatial_cluster/DLPFC/downstream.yaml")
print(f"  Slices: {len(SLICE_IDS)}, seeds: {SEEDS}")
print(f"  PCA_N={PCA_N}, spatial_weight={SPATIAL_WEIGHT}, adj_k={KNN_K}, resolution={LEIDEN_RES}")
print(f"  HVG_TOP_GENES={HVG_TOP_GENES}")
print(f"  Metrics: {list(METRIC_FUNCS.keys())}")
print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")


## Core Pipeline

Feature construction, KNN graph, and Leiden clustering come from the synthetic `spagcn_cluster.py`.


In [ ]:
# ══════════════════════════════════════════════════════════
# The following functions are imported directly from spagcn_cluster.py and are no longer redefined in the notebook:
#   spg.build_features(adata, n_components, spatial_weight, random_seed)
#   spg.knn_connectivities_sklearn(features, k, weighted)
#   spg.cluster_with_graph_leiden(graph, resolution, random_seed, fallback_features)
#   spg.cluster_with_leiden(features, k, resolution, random_seed)
# ══════════════════════════════════════════════════════════

def build_features_with_embedding(adata, embedding, seed=42):
    """
    Extension: PCA(50) + external embedding (e.g. GPFM 1024d) concatenation -> + spatial
    Concatenates the image embedding on top of spg.build_features, then performs standardization.
    """
    x = adata.X
    n = min(PCA_N, x.shape[1], max(1, int(x.shape[0]) - 1))
    if n >= 2:
        expr = (spg.TruncatedSVD(n_components=n, random_state=seed).fit_transform(x.astype("float32"))
                if sparse.issparse(x)
                else spg.PCA(n_components=n, random_state=seed).fit_transform(np.asarray(x, "float32")))
    else:
        expr = x.toarray() if sparse.issparse(x) else np.asarray(x, "float32")
    expr = np.hstack([expr, embedding])  # PCA(50) + GPFM(1024) = 1074d
    co = spg.StandardScaler().fit_transform(np.asarray(adata.obsm["spatial"], "float32")) * SPATIAL_WEIGHT
    return np.hstack([spg.StandardScaler().fit_transform(expr), co])


def compute_metrics(pred_labels, true_labels):
    """Compute all 7 clustering metrics, automatically filtering NaN."""
    tl = np.asarray(true_labels).astype(str)
    pl = np.asarray(pred_labels).astype(str)
    keep = ~pd.isna(tl)
    tl, pl = tl[keep], pl[keep]
    return {name: round(float(fn(tl, pl)), 4) for name, fn in METRIC_FUNCS.items()}


print("Helper functions ready")
print("  build_features / knn_connectivities_sklearn / cluster_with_graph_leiden → from spagcn_cluster.py")
print("  build_features_with_embedding → notebook extension (image embedding concatenation)")

## Load Precomputed Data

Slice expression, image embeddings, and SpaGCN labels.


In [ ]:
# (key, display_name, dim)
EMBED_CONFIG = [
    ("gpfm",     "GPFM",   1024),
    ("uni",      "UNI",    1024),
    ("uni2_h",   "UNI2-h", 1536),
    ("gigapath", "GigaPath", 1536),
]

# (key, display_name) for SpaGCN labels
SPAGCN_CONFIG = [
    ("gpfm",     "GPFM"),
    ("uni",      "UNI"),
    ("uni2_h",   "UNI2-h"),
    ("gigapath", "GigaPath"),
]

print("Loading embeddings...")
emb_data = {}
for key, name, dim in EMBED_CONFIG:
    emb_data[key] = {}
    for sid in SLICE_IDS:
        p = DATA_DIR / "pkl" / "embeddings" / f"visium_{sid}_allspot_{key}_{PKL_SUFFIX}.pkl"
        emb_data[key][sid] = torch.load(p, weights_only=True, map_location="cpu").numpy().astype('float32')
    print(f"  {name:20s} ({dim:4d}d): {len(emb_data[key])} slices")

print("Loading precomputed SpaGCN labels...")
spagcn_labels = {}
for key, name in SPAGCN_CONFIG:
    fname = DATA_DIR / "pkl" / "spagcn_refined" / f"{key}_{SPAGCN_SUFFIX}_update_adj_multi.pkl"
    with open(fname, "rb") as f:
        raw = pickle.load(f)
    spagcn_labels[f"SpaGCN+{name}"] = {sid: np.array(raw[sid]).astype(str) for sid in SLICE_IDS}
    print(f"  SpaGCN+{name:20s}: loaded {len(spagcn_labels[f'SpaGCN+{name}'])} slices")

## Run Comparison

Run all methods on each of the 12 DLPFC slices and evaluate them with 7 clustering metrics.


In [ ]:
# ══════════════════════════════════════════════════════════
# Load the h5ad preprocessed by synthetic (already QC + Normalize + global HVG)
# ══════════════════════════════════════════════════════════
PROCESSED_DIR = Path("../../synthetic/data/02_interim/common/DLPFC")

# Load the combined file to get the globally shared genes (matching the synthetic pipeline)
combined = ad.read_h5ad(PROCESSED_DIR / "processed_combined.h5ad")
COMMON_GENES = combined.var_names.tolist()
print(f"Global shared genes: {len(COMMON_GENES)}")

all_rows = []
for sid in SLICE_IDS:
    adata = ad.read_h5ad(PROCESSED_DIR / f"{sid}_processed.h5ad")

    # Align embeddings
    a_raw = ad.read_h5ad(DATA_DIR / "st" / f"{sid}_adata.h5ad")
    raw_name_to_idx = {f"{sid}_{name}": i for i, name in enumerate(a_raw.obs_names)}
    processed_names = set(adata.obs_names.astype(str))
    keep_idx = [i for name, i in raw_name_to_idx.items() if name in processed_names]
    for key, _, _ in EMBED_CONFIG:
        e = emb_data.get(key, {}).get(sid)
        if e is not None and e.shape[0] != adata.n_obs:
            emb_data[key][sid] = e[keep_idx]
    for full_method in list(spagcn_labels.keys()):
        labels = spagcn_labels[full_method].get(sid)
        if labels is not None and len(labels) != adata.n_obs:
            spagcn_labels[full_method][sid] = labels[keep_idx]

    # Unify the gene space: take the globally shared genes (replacing the original per-slice HVG)
    common_genes = [g for g in COMMON_GENES if g in adata.var_names]
    adata = adata[:, common_genes].copy()
    true_labels = adata.obs["label"]
    nc = N_CLUSTERS_BY_SLICE[sid]
    print(f"\n[{sid}] {adata.n_obs} spots, {adata.n_vars} HVGs, {nc} layers", flush=True)

    # ---- Method 1: PCA (synthetic default) ----
    fp = spg.build_features(adata, n_components=PCA_N, spatial_weight=SPATIAL_WEIGHT, random_seed=42)
    for s in SEEDS:
        graph = spg.build_backend_graph(fp, adata.obsm["spatial"], "pca_spatial_leiden", KNN_K, knn_backend=KNN_BACKEND)
        c = spg.cluster_with_graph_leiden(graph, resolution=LEIDEN_RES, random_seed=s, fallback_features=fp)
        all_rows.append({"Slice":sid,"Method":"PCA","Seed":s,"Feature":"PCA(50)+spatial", **compute_metrics(c,true_labels)})
        spg.write_spatial_cluster_result(
            adata, c, OUTPUT_DIR / "cluster_results" / f"{sid}_PCA_seed{s}",
            {"model": "PCA", "paradigm_family": "real_only", "variant_id": "real_only"},
            slice_id=sid, ablation="path_B", backend="pca_spatial_leiden", random_seed=s)
    ari1 = np.mean([r["ARI"] for r in all_rows if r["Slice"]==sid and r["Method"]=="PCA"])
    print(f"  [1] PCA:       ARI={ari1:.4f}  (config: downstream.yaml)", flush=True)

    # ---- Methods 2-5: PCA+GPFM/UNI/UNI2-h/GigaPath ----
    for idx, (key, name, dim) in enumerate(EMBED_CONFIG, start=2):
        fg = build_features_with_embedding(adata, emb_data[key][sid], seed=42)
        for s in SEEDS:
            graph = spg.build_backend_graph(fg, adata.obsm["spatial"], "pca_spatial_leiden", KNN_K, knn_backend=KNN_BACKEND)
            c = spg.cluster_with_graph_leiden(graph, resolution=EMBED_RESOLUTION, random_seed=s, fallback_features=fg)
            all_rows.append({"Slice":sid,"Method":f"PCA+{name}","Seed":s,"Feature":f"PCA(50)+{key}({dim})+spatial", **compute_metrics(c,true_labels)})
            spg.write_spatial_cluster_result(
                adata, c, OUTPUT_DIR / "cluster_results" / f"{sid}_PCA+{name}_seed{s}",
                {"model": f"PCA+{name}", "paradigm_family": "real_only", "variant_id": "real_only"},
                slice_id=sid, ablation="path_B", backend="pca_spatial_leiden", random_seed=s)
        ari = np.mean([r["ARI"] for r in all_rows if r["Slice"]==sid and r["Method"]==f"PCA+{name}"])
        print(f"  [{idx}] PCA+{name:20s} ARI={ari:.4f}", flush=True)

    # ---- Methods 6-9: SpaGCN+GPFM/UNI/UNI2-h/GigaPath (precomputed) ----
    for idx, (key, name) in enumerate(SPAGCN_CONFIG, start=6):
        full_method = f"SpaGCN+{name}"
        c = spagcn_labels[full_method][sid]
        all_rows.append({"Slice":sid,"Method":full_method,"Seed":-1,"Feature":f"GCN(DEC)+{key}+refine", **compute_metrics(c,true_labels)})
        spg.write_spatial_cluster_result(
                adata, np.array(c), OUTPUT_DIR / "cluster_results" / f"{sid}_{full_method}",
                {"model": full_method, "paradigm_family": "spagcn", "variant_id": "spagcn"},
                slice_id=sid, ablation="path_B", backend="spagcn", random_seed=-1)
        print(f"  [{idx}] {full_method:30s} ARI={all_rows[-1]['ARI']}", flush=True)

df = pd.DataFrame(all_rows)
for col in METRIC_FUNCS:
    df[col] = pd.to_numeric(df[col], errors="coerce")
print(f"\n{len(df)} result rows in total")


In [ ]:
metric_cols = list(METRIC_FUNCS)
seed_rows = df[df["Seed"]>=0]
spagcn_rows = df[df["Seed"]<0]

per_slice = pd.concat([
    seed_rows.groupby(["Slice","Method"])[metric_cols].mean().round(4),
    spagcn_rows.set_index(["Slice","Method"])[metric_cols]
]).sort_index()

print("="*100)
print("  Per slice x per method (averaged over seeds)")
print("="*100)
display(per_slice)


In [ ]:
# Each SpaGCN model has only a single run (no multiple seeds)
# Build one row for each SpaGCN model
spagcn_overall_rows = []
for method_name in spagcn_rows["Method"].unique():
    sub = spagcn_rows[spagcn_rows["Method"] == method_name][metric_cols]
    vals = sub.agg(["mean","std"]).round(4)  # (2, 7)
    row = vals.T.stack().to_frame().T  # (1, 14) with MultiIndex
    row.index = [method_name]
    spagcn_overall_rows.append(row)
spagcn_overall = pd.concat(spagcn_overall_rows)

overall = pd.concat([
    seed_rows.groupby("Method")[metric_cols].agg(["mean","std"]).round(4),
    spagcn_overall
])

In [ ]:
baseline_ari = seed_rows[seed_rows["Method"]=="PCA"].groupby("Slice")["ARI"].mean()
baseline_nmi = seed_rows[seed_rows["Method"]=="PCA"].groupby("Slice")["NMI"].mean()

print("="*100)
print("  ΔARI / ΔNMI vs PCA baseline — relative improvement")
print("="*100)
for method in df["Method"].unique():
    if method == "PCA": continue
    # Take each SpaGCN model from spagcn_rows
    if method.startswith("SpaGCN+"):
        sub = spagcn_rows[spagcn_rows["Method"] == method]
        ma = sub.groupby("Slice")["ARI"].mean()
        mn = sub.groupby("Slice")["NMI"].mean()
    else:
        ma = seed_rows[seed_rows["Method"]==method].groupby("Slice")["ARI"].mean()
        mn = seed_rows[seed_rows["Method"]==method].groupby("Slice")["NMI"].mean()
    if len(ma) > 0:
        da = (ma - baseline_ari).round(4)
        dn = (mn - baseline_nmi).round(4)
        print(f"\n{method}:")
        print(f"  ΔARI = {da.mean():+.4f} ± {da.std():.4f}")
        print(f"  ΔNMI = {dn.mean():+.4f} ± {dn.std():.4f}")


In [ ]:
df.to_csv(OUTPUT_DIR / "per_run_results.csv", index=False)
print(f"Per-spot clustering saved to {OUTPUT_DIR / 'cluster_results'}/")
per_slice.to_csv(OUTPUT_DIR / "per_slice_summary.csv")
overall.to_csv(OUTPUT_DIR / "overall_summary.csv")
print(f"Results saved to {OUTPUT_DIR.resolve()}/")
